---
## 8. Model Comparison Summary

Compare MNL (02), NL (03), and MXL (03b) on goodness-of-fit.
Which specification does the data support?

In [ ]:
# LR test using MNL LL from 02 (restricted model = MNL = σ=0)
# Note: MXL with σ=0 IS the MNL, so MNL LL is the restricted LL
LL_MNL = mnl_est["goodness_of_fit"]["ll_final"]
LL_MXL = ll_mxl

LR_MXL = -2 * (LL_MNL - LL_MXL)
df_mxl = len(RANDOM_ASC_MODES)  # 3 restricted params (σ at boundary)

# Boundary-corrected critical values (Gourieroux et al. 1982, Table 1)
# For 3 boundary params at α=0.05: mixture of χ² distributions
# Conservative: use χ²(df=3, α=0.05) = 7.81
from scipy.stats import chi2
crit_standard = chi2.ppf(0.95, df_mxl)
crit_boundary = chi2.ppf(0.90, df_mxl)  # more conservative for boundary

# Wald test on each σ (from Nelder-Mead — use finite-diff SE approximation)
# For Nelder-Mead, we approximate SE via Hessian at solution
def approx_se_from_solution(params, fn, args, eps=1e-3):
    """Crude SE approximation for Nelder-Mead output via Hessian of NLL."""
    k = len(params)
    try:
        H = np.zeros((k, k))
        for i in range(k):
            gi = lambda x, i=i: approx_fprime(x, fn, eps)[i]
            H[i] = approx_fprime(params, gi, eps)
        from numpy.linalg import inv, LinAlgError
        try:
            H_inv = inv(H)
            return np.sqrt(np.maximum(np.diag(H_inv), 0))
        except LinAlgError:
            return np.full(k, np.nan)
    except Exception:
        return np.full(k, np.nan)

se_mxl = approx_se_from_solution(result_mxl.x, mxl_nll_vectorized, (T, C, AVAIL, CHOICE, eta_draws))

print("=" * 70)
print("LR TEST — Mixed Logit vs MNL")
print("=" * 70)
print(f"  LL(MNL restricted, σ=0): {LL_MNL:.4f}")
print(f"  LL(MXL unrestricted):     {LL_MXL:.4f}")
print(f"  LR statistic:             {LR_MXL:.2f}")
print(f"  df:                       {df_mxl} (at boundary)")
print(f"  χ² crit (standard, α=0.05):   {crit_standard:.2f}")
print(f"  χ² crit (boundary, α=0.05):   {crit_boundary:.2f}")
print()
if LR_MXL < crit_boundary:
    print(f"  LR = {LR_MXL:.2f} < {crit_boundary:.2f} → FAIL TO REJECT H₀")
    print(f"  → No evidence of unobserved heterogeneity.")
    print(f"  → NL (03) is the appropriate specification.")
else:
    print(f"  LR = {LR_MXL:.2f} > {crit_boundary:.2f} → REJECT H₀ at α=0.05")
    print(f"  → Unobserved heterogeneity detected.")
    print(f"  → MXL is the appropriate specification for policy analysis.")

print()
print(f"Wald tests on individual σ:")
sigma_names = {"car": "σ_car", "moto": "σ_moto", "tj": "σ_tj"}
for i, m in enumerate(RANDOM_ASC_MODES):
    sigma_idx = 18 + i  # σ params are at positions 18, 19, 20
    se_s = se_mxl[sigma_idx] if not np.isnan(se_mxl[sigma_idx]) else np.nan
    t_stat = sigma_mxl[m] / se_s if se_s > 0 else 0
    sig = "***" if abs(t_stat) > 1.96 else "(not sig.)"
    print(f"  {sigma_names[m]:8s}: est = {sigma_mxl[m]:.5f}, SE ≈ {se_s:.5f}, t = {t_stat:+.2f} {sig}")

---
## 7. LR Test: MXL vs MNL/NL

H₀: σ_car = σ_moto = σ_tj = 0 (no unobserved heterogeneity)

The standard LR statistic is:

$$LR = -2 \cdot (LL_{restricted} - LL_{unrestricted})$$

**Boundary issue**: σ = 0 is on the boundary of the parameter space (σ ≥ 0).
The standard χ²(df=3) critical value is conservative. Following L07 and
Gourieroux et al. (1982), we use the boundary-corrected test:
χ²_{0.05} with df=3, boundary correction → conservative upper bound.

**Interpretation**:
- If LR is small and |t_σ| < 1.96 for all σ → fail to reject H₀ → NL is sufficient
- If LR is large or any |t_σ| > 1.96 → unobserved heterogeneity detected → MXL warranted

In [ ]:
# Starting values: MNL estimates + small initial σ
# Load MNL estimates as starting point
beta_time_start = mnl_est["beta_time"]  # from 02 notebook export
beta_cost_start = mnl_est["beta_cost"]
asc_start = mnl_est["asc"]
sigma_start = {"car": 0.01, "moto": 0.01, "tj": 0.01}

x0_mxl = pack_mxl(beta_time_start, beta_cost_start, asc_start, sigma_start)
ll_start = -mxl_nll_vectorized(x0_mxl, T, C, AVAIL, CHOICE, eta_draws)

print(f"Starting values (MNL estimates + σ=0.01):")
print(f"  LL = {ll_start:.4f}")
print(f"  σ_car = 0.01, σ_moto = 0.01, σ_tj = 0.01")
print(f"\nFitting Mixed Logit with {N_DRAWS} Halton draws...")
print(f"  Method: Nelder-Mead (gradient-free, handles simulation noise)")
print(f"  Max iterations: 2,000")

result_mxl = minimize(
    mxl_nll_vectorized,
    x0=x0_mxl,
    args=(T, C, AVAIL, CHOICE, eta_draws),
    method="Nelder-Mead",
    options={"maxiter": 2000, "xatol": 1e-4, "fatol": 1e-3, "adaptive": True},
)

ll_mxl = -result_mxl.fun
beta_time_mxl, beta_cost_mxl, asc_mxl, sigma_mxl = unpack_mxl(result_mxl.x)

print(f"\nConverged: {result_mxl.success}")
print(f"Message:   {result_mxl.message}")
print(f"Iterations: {result_mxl.nit}")
print(f"Function evals: {result_mxl.nfev}")
print(f"Final LL:  {ll_mxl:.4f}")
print(f"")
print(f"Random ASC SDs:")
for m in RANDOM_ASC_MODES:
    print(f"  σ_{m}: {sigma_mxl[m]:.6f}")

---
## 6. MXL Estimation

**Strategy**: Start from MNL estimates (σ = 0.01 small initial value). Use Nelder-Mead
(gradient-free, handles simulation noise). Expect DGP truth with σ=0 → 
σ estimates should remain small and not significant.

**Runtime**: ~2–5 minutes with Nelder-Mead on 5,000 persons × 200 draws.

In [ ]:
def mxl_nll_vectorized(params, T, C, AVAIL, CHOICE, eta_draws):
    """Vectorized simulated NLL — draws vectorized per person.

    For each person n:
      V_{n,j,r} = ASC_j + β_time_j·t_{n,j} + β_cost·c_{n,j} + σ_j·η_{r,k}
    Then average choice probability across r draws.

    Complexity: O(N × J × R) but with NumPy broadcasting on the R dimension.
    """
    beta_time, beta_cost, asc, sigma = unpack_mxl(params)
    N, J = T.shape
    R = eta_draws.shape[0]

    # Fixed utility (N × J)
    V_fixed = np.zeros((N, J), dtype=np.float64)
    for j, m in enumerate(MODE_LABELS):
        V_fixed[:, j] = asc[m] + beta_time[m] * T[:, j] + beta_cost * C[:, j]

    random_indices = np.array([MODE_LABELS.index(m) for m in RANDOM_ASC_MODES])  # [0, 1, 5]

    total_ll = 0.0

    for n in range(N):
        V_n = V_fixed[n].copy()  # (J,)
        avail_n = AVAIL[n]

        # Expand to (R × J): add random ASCs
        V_r = np.tile(V_n, (R, 1))  # (R, J)
        for k, j in enumerate(random_indices):
            V_r[:, j] += sigma[RANDOM_ASC_MODES[k]] * eta_draws[:, k]

        # Set unavailable modes
        V_r[:, ~avail_n] = -1e10

        # Log-sum-exp over available modes per draw
        V_max = V_r.max(axis=1, keepdims=True)
        expV = np.exp(V_r - V_max)
        denom = expV.sum(axis=1)  # (R,)

        # Probability of chosen alternative per draw
        P_draw = expV[:, CHOICE[n]] / denom  # (R,)
        P_avg = P_draw.mean()

        total_ll += np.log(max(P_avg, 1e-300))

    return -total_ll


# Test at truth (σ=0)
ll_true_v = -mxl_nll_vectorized(true_mxl, T, C, AVAIL, CHOICE, eta_draws)
print(f"Vectorized LL at DGP truth (σ=0): {ll_true_v:.4f}")
print(f"  MNL LL at true params (from 02):   -568.17")
print(f"  Gap (MXL - MNL): {ll_true_v - (-568.17):+.4f}")
print(f"  (Small gap expected — simulation noise with σ=0)")
print(f"\nEstimated time per function eval: ~{N_PERSONS * N_DRAWS * N_MODES / 1e6:.0f}M operations")

---
## 5. Vectorized Simulated Likelihood (Faster)

Loop over persons only — vectorize over draws. This is ~200× faster than
the naive person-inner loop over draws.

In [ ]:
def mxl_simulated_nll(params, T, C, AVAIL, CHOICE, eta_draws):
    """Simulated negative log-likelihood for Mixed Logit.

    Parameters
    ----------
    params : np.array (21,)
        Packed MXL parameters
    T, C : np.array (N × 9)
        Time and cost matrices
    AVAIL : np.array (N × 9)
        Availability mask
    CHOICE : np.array (N,)
        Chosen mode indices (0-indexed)
    eta_draws : np.array (R × 3)
        Halton draws for standard normal (car, moto, tj)

    Returns
    -------
    float : negative simulated log-likelihood
    """
    beta_time, beta_cost, asc, sigma = unpack_mxl(params)
    N, J = T.shape
    R = len(eta_draws)

    # Pre-compute the fixed part of V (no random component)
    V_fixed = np.zeros((N, J), dtype=np.float64)
    for j, m in enumerate(MODE_LABELS):
        V_fixed[:, j] = asc[m] + beta_time[m] * T[:, j] + beta_cost * C[:, j]
    V_fixed[~AVAIL] = -1e10  # effectively -inf for numerical stability

    # Random ASC indices in MODE_LABELS
    random_indices = np.array([MODE_LABELS.index(m) for m in RANDOM_ASC_MODES])  # [0, 1, 5]

    # For each person, compute average choice probability across draws
    log_prob_sum = 0.0

    for n in range(N):
        V_n = V_fixed[n]  # (9,) fixed utility
        avail_n = np.isfinite(V_n)
        P_n = 0.0

        for r in range(R):
            V_r = V_n.copy()
            # Add random ASC: σ_i × η_{i,r}
            for k, j in enumerate(random_indices):
                V_r[j] += sigma[RANDOM_ASC_MODES[k]] * eta_draws[r, k]
            V_r[~avail_n] = -1e10
            V_max = V_r[avail_n].max()
            expV = np.exp(V_r - V_max)
            denom = expV.sum()
            P_n += expV[CHOICE[n]] / denom

        P_n /= R
        log_prob_sum += np.log(max(P_n, 1e-300))

    return -log_prob_sum


def null_ll():
    """LL₀ — equal probability across available alternatives."""
    n_avail = AVAIL.sum(axis=1)
    return float(np.sum(np.log(1.0 / n_avail)))


LL_NULL = null_ll()
print(f"MXL simulated likelihood function defined.")
print(f"Null LL (equal share): {LL_NULL:.2f}")
print(f"N draws: {N_DRAWS}, N persons: {N_PERSONS}")

# Quick sanity check at true parameters (σ=0 → should ≈ MNL LL)
true_sigma = {"car": 0.0, "moto": 0.0, "tj": 0.0}
true_mxl = pack_mxl(TRUE_DGP["beta_time"], TRUE_DGP["beta_cost"], TRUE_DGP["asc"], true_sigma)
ll_true_mxl = -mxl_simulated_nll(true_mxl, T, C, AVAIL, CHOICE, eta_draws)
print(f"LL at DGP truth (σ=0): {ll_true_mxl:.4f}")
print(f"  MNL LL at truth:     {-mnl_est['goodness_of_fit']['ll_final'] if False else 'see 02 notebook'}")
print(f"  (Should be close to MNL LL — σ=0 reduces MXL → MNL)")

---
## 4. Simulated Log-Likelihood

For each Halton draw r, compute the MNL probability with ASC_i + σ_i · η_{i,r}:

$$P_n(i \mid \eta_r) = \frac{\exp(V_{in}(\eta_r))}{\sum_j \exp(V_{jn}(\eta_r))}$$

Then average over draws:

$$P_n(i) = \frac{1}{R} \sum_{r=1}^{R} P_n(i \mid \eta_r)$$

The simulated log-likelihood is: $\mathcal{L}_{sim} = \sum_n \log P_n(i_n)$

**Important**: The simulated LL is Jensen-biased downward (log of average < average of log).
This means MXL LL can appear lower than MNL LL even when correctly specified.
The LR test accounts for this — but the bias is small with 200 Halton draws.

In [ ]:
# Parameter packing for MXL
ASC_MODES_ESTIMABLE = [m for m in MODE_LABELS if m != REF_MODE]  # 8 ASCs estimable

def pack_mxl(beta_time, beta_cost, asc_dict, sigma_dict):
    """Pack MXL parameters: 9 β_time + 1 β_cost + 8 ASCs + 3 σ."""
    p = []
    for m in MODE_LABELS:
        p.append(beta_time[m])
    p.append(beta_cost)
    for m in ASC_MODES_ESTIMABLE:
        p.append(asc_dict[m])
    for m in RANDOM_ASC_MODES:
        p.append(abs(sigma_dict[m]))  # ensure positive SD
    return np.array(p)


def unpack_mxl(params):
    """Unpack MXL parameter vector."""
    beta_time = {m: params[i] for i, m in enumerate(MODE_LABELS)}
    beta_cost = params[9]
    asc = {REF_MODE: 0.0}
    for j, m in enumerate(ASC_MODES_ESTIMABLE):
        asc[m] = params[10 + j]
    sigma = {}
    for k, m in enumerate(RANDOM_ASC_MODES):
        sigma[m] = abs(params[18 + k])
    return beta_time, beta_cost, asc, sigma


N_PARAMS_MXL = N_MODES + 1 + (N_MODES - 1) + len(RANDOM_ASC_MODES)  # 9+1+8+3 = 21
print(f"MXL parameters: {N_PARAMS_MXL}  (MNL had 18)")
print(f"  9 β_time + 1 β_cost + 8 ASCs + 3 σ_ASC")

---
## 3. Mixed Logit Parameter Vector

**21 parameters** (18 MNL + 3 σ_ASC):

```
[β_time_car ... β_time_mrt,   (9 params)
 β_cost,                       (1 param)
 ASC_car, ASC_moto, ASC_4wrh, ASC_2wrh, ASC_tj, ASC_royal, ASC_lrt, ASC_mrt,  (8 params; KRL=0)
 σ_car, σ_moto, σ_tj]          (3 random ASC SDs)
```

In [ ]:
def halton_sequence(n, base=2):
    """Generate Halton sequence of length n in (0, 1) using given base."""
    seq = np.zeros(n)
    num = np.arange(1, n + 1)
    denom = 1
    while np.any(num > 0):
        denom *= base
        seq += (num % base) / denom
        num //= base
    return seq


N_DRAWS = 200
# 3 random ASCs → need 3 independent Halton dimensions (primes 2, 3, 5)
halton_dims = []
for dim, base in enumerate([2, 3, 5]):
    h = halton_sequence(N_DRAWS, base=base)
    # Convert to standard normal via inverse CDF
    eta = snorm.ppf(np.clip(h, 1e-10, 1 - 1e-10))
    halton_dims.append(eta.astype(np.float32))

eta_draws = np.column_stack(halton_dims)  # (N_DRAWS × 3)

print(f"Halton draws: {N_DRAWS} draws × {len(halton_dims)} dimensions")
print(f"  Base 2 (σ_car):  first 5 = {halton_dims[0][:5]}")
print(f"  Base 3 (σ_moto): first 5 = {halton_dims[1][:5]}")
print(f"  Base 5 (σ_tj):   first 5 = {halton_dims[2][:5]}")
print(f"  Coverage check: min={eta_draws.min():.2f}, max={eta_draws.max():.2f}")

---
## 2. Halton Draws for Simulated Likelihood

Halton sequences are quasi-random — they cover the [0,1] interval more evenly
than pseudo-random draws, reducing the number of draws needed for stable integration.

**Train (2009)**: 125–200 Halton draws ≈ 1,000 pseudo-random in coverage.
Ilahi (2021) uses 1,000 Halton draws on 52K observations (pooled SP/RP).
For 5,000 synthetic obs, 200 draws are sufficient.

In [ ]:
# Assemble data matrices
time_cols = [f"time_{m}" for m in MODE_LABELS]
cost_cols = [f"cost_{m}" for m in MODE_LABELS]

T = df_persons[time_cols].values  # time in minutes (N × 9)
C = df_persons[cost_cols].values  # cost in Thousand IDR (N × 9)
AVAIL = ~np.isnan(T)              # availability mask (N × 9)

# Compute DGP utility
V_dgp = np.zeros((N_PERSONS, N_MODES))
for j, m in enumerate(MODE_LABELS):
    V_dgp[:, j] = TRUE_DGP["asc"][m] + TRUE_DGP["beta_time"][m] * T[:, j] + TRUE_DGP["beta_cost"] * C[:, j]
V_dgp[~AVAIL] = -np.inf

# Generate Gumbel noise and choices
U = rng.uniform(1e-12, 1 - 1e-12, size=(N_PERSONS, N_MODES))
epsilon = -np.log(-np.log(U))
U_total = V_dgp + epsilon
U_total[np.isneginf(V_dgp)] = -np.inf

chosen_idx = np.argmax(U_total, axis=1)
CHOICE = chosen_idx  # 0-indexed
df_persons["choice_idx"] = CHOICE

# Choice shares
choice_shares = pd.Series([MODE_LABELS[i] for i in CHOICE]).value_counts(normalize=True).reindex(MODE_LABELS, fill_value=0)
print("Synthetic choice distribution:\n")
for m in MODE_LABELS:
    bar = "█" * int(choice_shares[m] * 100)
    print(f"  {m:6s}: {choice_shares[m]*100:5.1f}% {bar}")

# ASC modes (random ASCs on car, moto, tj)
RANDOM_ASC_MODES = ["car", "moto", "tj"]  # 3 random ASCs
FIXED_ASC_MODES = [m for m in MODE_LABELS if m not in RANDOM_ASC_MODES and m != REF_MODE]
# Fixed ASCs: 4wrh, 2wrh, royal, lrt, mrt (5 modes; KRL=0)

print(f"\nRandom ASCs on: {RANDOM_ASC_MODES}")
print(f"Fixed ASCs on:  {FIXED_ASC_MODES}")
print(f"Reference:      {REF_MODE} (ASC=0)")

## 1. Generate Synthetic Choices (same DGP as 02)

Choices are generated from the fixed-parameter DGP — same as notebook 02.
The DGP has NO built-in heterogeneity, so the Mixed Logit should correctly
find that random ASCs are not warranted.

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize, approx_fprime
from scipy.stats import norm as snorm
from pathlib import Path
import json

np.random.seed(2026)
rng = np.random.default_rng(2026)

# Paths
NOTEBOOK_DIR = Path.cwd()
DATA_DIR = NOTEBOOK_DIR / "data"

# Load data
df_persons = pd.read_csv(DATA_DIR / "persons_jkt.csv")

N_PERSONS = len(df_persons)

# Mode definitions
MODE_LABELS = ["car", "moto", "4wrh", "2wrh", "krl", "tj", "royal", "lrt", "mrt"]
N_MODES = len(MODE_LABELS)
REF_MODE = "krl"

# Load DGP parameters (same as 02)
TRUE_DGP = {
    "asc": {
        "krl": 0.00, "car": 0.90, "moto": 1.20, "2wrh": 1.10,
        "4wrh": 0.50, "mrt": 0.30, "royal": 0.05, "lrt": -0.10, "tj": -0.30,
    },
    "beta_time": {
        "car": -0.60, "moto": -2.34, "4wrh": -3.49, "2wrh": -5.10,
        "krl": -2.72, "tj": -1.07, "royal": -1.30, "lrt": -2.37, "mrt": -2.98,
    },
    "beta_cost": -1.42,
}

# Load MNL estimates from 02 for comparison
with open(DATA_DIR / "mnl_estimates.json") as f:
    mnl_est = json.load(f)

print(f"Loaded: {N_PERSONS} persons, {N_MODES} modes")
print(f"MNL LL = {mnl_est['goodness_of_fit']['ll_final']:.2f}")
print(f"MNL ρ² = {mnl_est['goodness_of_fit']['rho2']:.4f}")

**Trans-Eng Final Project — Hiroshima University AY2026**  
**Depends on**: `01_data_prep.ipynb`, `02_mnl_estimation.ipynb`, `03_nl_estimation.ipynb`  
**Spec reference**: `trans-eng-mixed-logit-addition.md`  
**Code reference**: `notebooks/trans-eng-lectures/L07/L07_estimation_lab.ipynb` Tasks 3 + 3.5